# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset (ordered logistic regression outputs for rangeland knowledge adoption predictors) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. Here, we'll list all record sets, and for each, enumerate its fields and columns by `@id`.

In [ ]:
# List all Record Sets and their fields/columns by @id
record_sets = list(dataset.record_sets.keys())
print("Available record sets:")
for record_set_id in record_sets:
    print(f"\nRecord Set @id: {record_set_id}")
    rs = dataset.record_sets[record_set_id]
    if hasattr(rs, 'fields') and rs.fields:
        print("Fields:")
        for field_id in rs.fields:
            print(f"  - {field_id}")
    if hasattr(rs, 'columns') and rs.columns:
        print("Columns:")
        for col_id in rs.columns:
            print(f"  - {col_id}")


In [ ]:
# Preview a few records from one of the record sets (select the first available if multiple exist)
if record_sets:
    preview_record_set = record_sets[0]
    print(f"\nExample records from record set: {preview_record_set}")
    for i, record in enumerate(dataset.records(record_set=preview_record_set)):
        if i >= 3:
            break
        print(record)
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field or column `@id`s from the overview above.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

if record_sets:
    first_rs = record_sets[0]
    print(f"Columns available in record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No record sets to extract data from.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records by a numeric field.
- Normalizing numeric field.
- Grouping by a categorical or grouping field.

All field/column access uses `@id` as per the record set schema.

In [ ]:
# Example EDA on the first available record set
import numpy as np

# Identify numeric and group fields by displaying column list
if record_sets:
    rs_id = record_sets[0]
    df = dataframes[rs_id]
    print("Available columns:", df.columns.tolist())

    # Try to auto-select a likely numeric field
    numeric_field_candidates = [col for col in df.columns if ('coefficient' in col.lower() or 'p' in col.lower() or 'std' in col.lower()) and pd.api.types.is_numeric_dtype(df[col])]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
    else:
        # Fallback: first numeric-type column
        numeric_field_id = next((col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])), None)

    print("Using numeric field for filtering and normalization:", numeric_field_id)

    # Choose group field (categorical variable), e.g., if field contains 'variable' or 'group' in its name
    group_field_candidates = [col for col in df.columns if 'variable' in col.lower() or 'group' in col.lower()]
    group_field = group_field_candidates[0] if group_field_candidates else df.columns[0]
    print("Using group field for grouping:", group_field)

    # Filtering (use a quantile as threshold if values are not known)
    if numeric_field_id is not None:
        # Remove NaN first
        valid_df = df.dropna(subset=[numeric_field_id])
        if not valid_df.empty:
            threshold = valid_df[numeric_field_id].quantile(0.75)
            filtered_df = valid_df[valid_df[numeric_field_id] > threshold].copy()
            print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
            display(filtered_df.head())

            # Normalization
            mean = filtered_df[numeric_field_id].mean()
            std = filtered_df[numeric_field_id].std()
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Group by
            if group_field in filtered_df.columns and pd.api.types.is_string_dtype(filtered_df[group_field]):
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
                print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
                display(grouped_df.head())
        else:
            print(f"No valid data in numeric field {numeric_field_id}.")
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No record sets found; unable to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All fields referenced use their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# Example: histogram and boxplot of the selected numeric field, grouped by the grouping field
if record_sets and numeric_field_id is not None and group_field is not None:
    plt.figure(figsize=(10, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color='teal')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping field is not too high-cardinality
    if group_field in df.columns and df[group_field].nunique() <= 10:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR² dataset using the Croissant schema and the `mlcroissant` library.

- We accessed metadata via schema and reviewed record sets and their fields/columns (by `@id`).
- Data from record sets was loaded into pandas DataFrames via their `@id`.
- Simple EDA and normalization was performed using field `@id`s only.
- Visualizations illustrated numeric field distributions and grouped summary views.

Continue with domain-specific analysis or modeling as needed using the structured access enabled by Croissant and `mlcroissant`.
